# Lecture 11 — Clustering: K-Means and the EM Algorithm

**Practices covered:** L11 (K-Means), L12 (EM / Gaussian Mixture Models), L13 (Spam Filter)

---

## Unsupervised Learning: No Labels

Everything so far was **supervised**: you had input–output pairs $(x, y)$ and learned to predict $y$ from $x$.

**Unsupervised learning:** you only have inputs $\{x_1, x_2, \ldots, x_N\}$ — **no labels**.

**Clustering** finds hidden structure: group similar points together without being told what the groups are.

**Real examples:**
- Group customers by purchasing behavior (without pre-defined groups)
- Identify topics in a document collection
- Compress an image by replacing each pixel color with the nearest of $K$ representative colors
- Find gene expression patterns in biology

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
%matplotlib inline

# Generate 3-cluster data
np.random.seed(42)
means = np.array([[0, 0], [5, 5], [0, 9]])
clusters = [np.random.randn(60, 2) + mu for mu in means]
X = np.vstack(clusters)
true_labels = np.array([0]*60 + [1]*60 + [2]*60)

plt.figure(figsize=(6, 5))
plt.scatter(X[:,0], X[:,1], c='gray', alpha=0.6, s=20)
plt.title('Unlabeled data — can you see 3 groups?')
plt.xlabel('x₁'); plt.ylabel('x₂'); plt.show()

## Part 1 — K-Means: Hard Assignment Clustering

**Algorithm:**
1. Initialize $K$ cluster centers (centroids) — randomly from data points
2. **E-step (Assign):** For each point, find the nearest centroid → assign the point to that cluster
3. **M-step (Update):** For each cluster, compute the mean of all assigned points → new centroid
4. Repeat steps 2–3 until centroids don't move

**What K-means minimizes:**
$$L = \sum_{n=1}^{N} \sum_{k=1}^{K} r_{nk} \|\mathbf{x}_n - \boldsymbol{\mu}_k\|^2$$

where $r_{nk} = 1$ if point $n$ belongs to cluster $k$, else 0. This is the total squared distance from points to their assigned centroids.

In [ ]:
def findClosestCentroids(X, centroids):
    """
    Assign each point to nearest centroid.
    Returns idx: shape (m,) with centroid index for each point.
    Matches the implementation in Practice L11.
    """
    K = centroids.shape[0]
    idx = np.zeros(X.shape[0], dtype=int)
    
    for i in range(X.shape[0]):
        min_dist = np.inf
        for k in range(K):
            diff = X[i] - centroids[k]
            dist = diff @ diff          # squared Euclidean distance
            if dist < min_dist:
                min_dist = dist
                idx[i] = k
    return idx

def computeCentroids(X, idx, K):
    """Recompute centroids as mean of assigned points."""
    n = X.shape[1]
    centroids = np.zeros((K, n))
    for k in range(K):
        cluster_k = X[idx == k]
        if len(cluster_k) > 0:
            centroids[k] = cluster_k.mean(axis=0)
    return centroids

def runkMeans(X, K, max_iters=20, seed=42):
    """Full K-means algorithm."""
    np.random.seed(seed)
    # Initialize: pick K random data points as centroids
    init_idx = np.random.permutation(len(X))[:K]
    centroids = X[init_idx].copy()
    
    history = [centroids.copy()]
    for _ in range(max_iters):
        idx = findClosestCentroids(X, centroids)
        centroids = computeCentroids(X, idx, K)
        history.append(centroids.copy())
        
    return centroids, idx, history

centroids, assignments, history = runkMeans(X, K=3, max_iters=10)
print("Final centroids:")
print(centroids.round(2))
print("True means:")
print(means)

In [ ]:
# Visualize convergence step by step
n_steps = min(5, len(history))
fig, axes = plt.subplots(1, n_steps, figsize=(3.5*n_steps, 4))
colors = ['#E74C3C', '#3498DB', '#2ECC71']

for step in range(n_steps):
    ax = axes[step]
    c = history[step]
    idx_step = findClosestCentroids(X, c)
    
    for k in range(3):
        pts = X[idx_step == k]
        ax.scatter(pts[:,0], pts[:,1], c=colors[k], alpha=0.4, s=15)
        ax.scatter(c[k,0], c[k,1], c=colors[k], s=200, marker='X', edgecolors='black', linewidths=1.5)
    
    ax.set_title(f'Iteration {step}')
    ax.set_xlim(-4, 9); ax.set_ylim(-4, 13)

plt.suptitle('K-means convergence: X = centroids', y=1.02)
plt.tight_layout(); plt.show()

## Part 2 — K-Means Pitfall: Feature Scale Matters!

K-means uses **Euclidean distance**. If feature 1 ranges from 0–1000 and feature 2 ranges from 0–1, the algorithm will almost entirely ignore feature 2 — a 1-unit change in feature 1 dominates.

**Always standardize features before K-means** (exactly like regression).

In [ ]:
# Demonstrate scale sensitivity
np.random.seed(0)
income = np.array([30000, 31000, 32000, 90000, 91000, 92000], dtype=float)  # scale 0-100k
age    = np.array([25,    45,    55,    28,    47,    60],    dtype=float)  # scale 0-80
X_scale = np.column_stack([income, age])
labels_true = np.array([0, 0, 0, 1, 1, 1])  # true: 2 groups by income

# Without scaling
c_raw, idx_raw, _ = runkMeans(X_scale, K=2, max_iters=20)

# With scaling
X_std = (X_scale - X_scale.mean(axis=0)) / X_scale.std(axis=0)
c_std, idx_std, _ = runkMeans(X_std, K=2, max_iters=20)

print("Without scaling — assignments:", idx_raw)
print("True labels:                   ", labels_true)
print("Match:", np.mean(idx_raw == labels_true) in [np.mean(idx_raw==labels_true), np.mean(idx_raw==(1-labels_true))])
print()
# Standardize idx_std labels (may be 0/1 swapped)
match_std = max(np.mean(idx_std == labels_true), np.mean(idx_std == (1-labels_true)))
print("With scaling — accuracy:", f"{match_std*100:.0f}%")

## Part 3 — Application: Image Compression

An image is just a matrix of (R,G,B) pixels — each pixel is a point in 3D color space. K-means can cluster colors:
- Find $K$ representative colors (centroids)
- Replace each pixel with its nearest representative color
- Store only $K$ colors + assignment index per pixel → much less data

In [ ]:
# Simulate image compression with a synthetic gradient image
np.random.seed(7)
H, W = 40, 60
# Create image with smooth gradients
x_img = np.linspace(0, 1, W)
y_img = np.linspace(0, 1, H)
XX, YY = np.meshgrid(x_img, y_img)
R = XX; G = YY; B = 1 - XX*YY
image = np.stack([R, G, B], axis=2)  # (H, W, 3)
pixels = image.reshape(-1, 3)         # (H*W, 3) — each row is a pixel color

# Compress with K=8 colors
K_img = 8
centroids_img, assignments_img, _ = runkMeans(pixels, K=K_img, max_iters=20)

# Reconstruct: replace each pixel with its centroid color
reconstructed = centroids_img[assignments_img]
image_compressed = reconstructed.reshape(H, W, 3)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(np.clip(image, 0, 1)); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(np.clip(image_compressed, 0, 1)); axes[1].set_title(f'K={K_img} colors'); axes[1].axis('off')

# Show the K representative colors
color_bar = centroids_img[np.newaxis, :, :].clip(0, 1)
axes[2].imshow(color_bar, aspect='auto')
axes[2].set_title(f'{K_img} representative colors')
axes[2].set_yticks([]); axes[2].set_xticks(range(K_img))

original_size = H * W * 3 * 8  # bits
compressed_size = H*W * np.ceil(np.log2(K_img)) + K_img*3*8
print(f"Original: {original_size/8:.0f} bytes")
print(f"Compressed: {compressed_size/8:.0f} bytes  ({compressed_size/original_size*100:.1f}% of original)")
plt.tight_layout(); plt.show()

## Part 4 — Gaussian Mixture Models: Soft Clustering

K-means makes **hard assignments**: each point belongs to exactly one cluster.

**Problem:** What about a point exactly between two clusters? K-means must choose one — but realistically it's 50% likely to belong to either.

**Gaussian Mixture Model (GMM):** Each cluster is a Gaussian distribution. A point can partially belong to multiple clusters.

$$p(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$$

- $\pi_k$ = mixing weight (how common is cluster $k$): $\sum_k \pi_k = 1$
- $\boldsymbol{\mu}_k$ = mean of cluster $k$
- $\boldsymbol{\Sigma}_k$ = covariance (shape/spread) of cluster $k$

In [ ]:
# Visualize: GMM can model differently shaped and sized clusters
np.random.seed(1)
K = 3
true_means = [np.array([0, 0]), np.array([5, 1]), np.array([2, 6])]
true_covs  = [np.array([[1.5, 0.5], [0.5, 0.5]]),
              np.array([[0.5, 0.0], [0.0, 2.0]]),
              np.array([[0.8, -0.4], [-0.4, 0.8]])]
true_pis   = [0.3, 0.4, 0.3]

X_gmm = np.vstack([
    np.random.multivariate_normal(mu, cov, int(pi*200))
    for mu, cov, pi in zip(true_means, true_covs, true_pis)
])

xx, yy = np.meshgrid(np.linspace(-4, 9, 100), np.linspace(-4, 11, 100))
grid = np.column_stack([xx.ravel(), yy.ravel()])

plt.figure(figsize=(7, 6))
plt.scatter(X_gmm[:,0], X_gmm[:,1], c='gray', alpha=0.4, s=15, label='Data')

colors = ['red', 'blue', 'green']
for mu, cov, pi, color, lbl in zip(true_means, true_covs, true_pis, colors, ['Cluster A', 'Cluster B', 'Cluster C']):
    z = multivariate_normal.pdf(grid, mean=mu, cov=cov).reshape(100, 100)
    plt.contour(xx, yy, z, levels=3, colors=[color], alpha=0.7)
    plt.scatter(*mu, c=color, s=200, marker='X', edgecolors='black', label=f'{lbl} (π={pi})')

plt.xlabel('x₁'); plt.ylabel('x₂')
plt.title('Gaussian Mixture Model: each cluster is a Gaussian\n(contours show the Gaussian shapes)')
plt.legend(); plt.show()

## Part 5 — The EM Algorithm: Iterative Learning with Hidden Variables

We can't use plain gradient descent to fit a GMM because the cluster labels are **hidden** (latent) — we don't know which cluster each point belongs to.

**EM (Expectation-Maximization)** solves this by alternating:

**E-step (Expectation):** Given current parameters, compute the *responsibility* $\gamma_{nk}$ — the probability that point $n$ belongs to cluster $k$:

$$\gamma_{nk} = \frac{\pi_k \, \mathcal{N}(\mathbf{x}_n \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)}{\sum_{j=1}^K \pi_j \, \mathcal{N}(\mathbf{x}_n \mid \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)}$$

**M-step (Maximization):** Given responsibilities, update parameters using weighted statistics:

$$N_k = \sum_n \gamma_{nk}, \quad \pi_k = \frac{N_k}{N}, \quad \boldsymbol{\mu}_k = \frac{1}{N_k}\sum_n \gamma_{nk} \mathbf{x}_n, \quad \boldsymbol{\Sigma}_k = \frac{1}{N_k}\sum_n \gamma_{nk}(\mathbf{x}_n-\boldsymbol{\mu}_k)(\mathbf{x}_n-\boldsymbol{\mu}_k)^T$$

In [ ]:
# Full EM for Gaussian Mixture Models
def initialize_gmm(X, K, seed=0):
    np.random.seed(seed)
    m, n = X.shape
    idx = np.random.choice(m, K, replace=False)
    mu = X[idx].copy().astype(float)
    sigma = [np.eye(n)] * K         # start with identity covariance
    pi = np.ones(K) / K             # equal mixing weights
    return pi, mu, sigma

def e_step(X, pi, mu, sigma):
    """Compute responsibilities γ[n,k] = P(cluster k | point n)."""
    K = len(pi)
    m = X.shape[0]
    gamma = np.zeros((m, K))
    
    for k in range(K):
        # Evaluate Gaussian density for cluster k at each point
        gamma[:, k] = pi[k] * multivariate_normal.pdf(X, mean=mu[k], cov=sigma[k])
    
    # Normalize: each row sums to 1 (responsibilities must sum to 1 per point)
    row_sums = gamma.sum(axis=1, keepdims=True)
    gamma /= (row_sums + 1e-300)     # avoid division by zero
    return gamma

def m_step(X, gamma):
    """Update parameters given responsibilities."""
    m, n = X.shape
    K = gamma.shape[1]
    
    Nk = gamma.sum(axis=0)           # effective cluster sizes: shape (K,)
    pi = Nk / m                       # mixing weights
    mu = (gamma.T @ X) / Nk[:, np.newaxis]  # weighted means: (K, n)
    
    sigma = []
    for k in range(K):
        diff = X - mu[k]              # (m, n)
        # Weighted covariance: sum_n γ[n,k] * (x_n - μ_k)(x_n - μ_k)^T
        cov_k = (gamma[:, k:k+1] * diff).T @ diff / Nk[k]
        cov_k += 1e-6 * np.eye(n)    # regularize to prevent singular covariance
        sigma.append(cov_k)
    
    return pi, mu, sigma

def run_em(X, K, max_iters=30, seed=0):
    """Run EM for Gaussian Mixture Model."""
    pi, mu, sigma = initialize_gmm(X, K, seed)
    log_likelihoods = []
    
    for iteration in range(max_iters):
        gamma = e_step(X, pi, mu, sigma)    # E-step
        pi, mu, sigma = m_step(X, gamma)    # M-step
        
        # Monitor log-likelihood (should increase monotonically)
        ll = sum(np.log(sum(pi[k] * multivariate_normal.pdf(X, mu[k], sigma[k])
                            for k in range(K)) + 1e-300))
        log_likelihoods.append(ll)
    
    return pi, mu, sigma, gamma, log_likelihoods

# Test on our 3-cluster data
pi_fit, mu_fit, sigma_fit, gamma_fit, lls = run_em(X, K=3, max_iters=50)

print("EM converged. Learned parameters:")
for k in range(3):
    print(f"  Cluster {k}: weight={pi_fit[k]:.2f}, mean={mu_fit[k].round(2)}")

plt.figure(figsize=(6, 3))
plt.plot(lls, 'b-o', markersize=4)
plt.xlabel('EM iteration'); plt.ylabel('Log-likelihood')
plt.title('EM always improves log-likelihood (monotone convergence)')
plt.grid(True, alpha=0.3); plt.show()

In [ ]:
# Visualize: soft assignments from EM vs hard assignments from K-means
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors_rgb = np.array([[0.9,0.2,0.2], [0.2,0.4,0.9], [0.2,0.8,0.4]])

# K-means: hard assignments
_, km_assignments, _ = runkMeans(X, K=3)
for k in range(3):
    pts = X[km_assignments == k]
    axes[0].scatter(pts[:,0], pts[:,1], c=[colors_rgb[k]], alpha=0.6, s=20)
axes[0].set_title('K-means: HARD assignments\n(each point 100% one cluster)')
axes[0].set_xlabel('x₁'); axes[0].set_ylabel('x₂')

# EM/GMM: soft assignments — color = blend of cluster colors
blended_colors = gamma_fit @ colors_rgb  # (m, 3) — each point's color is weighted mix
axes[1].scatter(X[:,0], X[:,1], c=blended_colors, s=20)
for k in range(3):
    axes[1].scatter(*mu_fit[k], c=[colors_rgb[k]], s=200, marker='X',
                    edgecolors='black', linewidths=2)
axes[1].set_title('EM/GMM: SOFT assignments\n(uncertainty shown as mixed color)')
axes[1].set_xlabel('x₁')

# Annotate a borderline point
border_idx = np.argmax(np.min(gamma_fit, axis=1))  # most uncertain point
axes[1].annotate(f'Uncertain point\n({gamma_fit[border_idx].round(2)})',
                 xy=X[border_idx], xytext=(X[border_idx,0]+1, X[border_idx,1]-1),
                 arrowprops=dict(arrowstyle='->', color='black'), fontsize=8)

plt.tight_layout(); plt.show()

## Part 6 — K-means vs EM: When to Use Which

| | K-means | EM / GMM |
|---|---|---|
| Assignment | Hard (0 or 1) | Soft (probability) |
| Cluster shape | Only spherical (Euclidean) | Any shape (via covariance) |
| Speed | Fast | Slower (covariance computation) |
| Uncertainty | None | Explicit probability |
| Convergence | Local minimum of distortion | Local maximum of likelihood |
| Use when | Fast grouping, image compression | Probabilistic model needed, overlapping clusters |

**The connection:** K-means is a special case of EM where covariances are fixed as identity matrices and assignments become hard (0 or 1) instead of soft.

## Part 7 — Spam Filter (Practice L13): Putting It Together

The spam filter is a **supervised classification** problem (not clustering), but it uses the same ideas from the course:

1. **Text preprocessing** → convert email to numerical features
2. **Feature representation** → bag of words (binary vector)
3. **Classifier** → SVM (or logistic regression)

The key insight: represent each email as a vector of 0s and 1s — 1 if word $j$ appears in the email, 0 otherwise.

In [ ]:
import re

def process_email(text):
    """Email preprocessing pipeline from Practice L13."""
    text = text.lower()
    text = re.sub(r'<[^<>]+>', ' ', text)          # remove HTML tags
    text = re.sub(r'(http|https)://\S+', 'httpaddr', text)  # normalize URLs
    text = re.sub(r'[\d]+', 'number', text)         # normalize numbers
    text = re.sub(r'[\$]+', 'dollar', text)         # normalize currency
    text = re.sub(r'[^a-z ]', ' ', text)            # remove punctuation
    words = [w for w in text.split() if len(w) > 2]
    return words

spam_email = """
    <html>Buy NOW! Get FREE medication at the LOWEST prices ever!
    Click httpaddr to claim your $99.99 prize! LIMITED TIME: call 1800-123!!!
    </html>
"""
ham_email = """
    Hi John, I hope this email finds you well. 
    Could we schedule a meeting for Thursday at 3pm to discuss the project?
    Please let me know if that works for you. Best regards, Alice.
"""

print("Spam words:", process_email(spam_email))
print("Ham words: ", process_email(ham_email))

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

# Minimal vocabulary
vocab = ['buy', 'free', 'medication', 'limited', 'click', 'prize',
         'meeting', 'schedule', 'project', 'regards', 'dollar', 'now', 'lowest']

def email_to_features(email, vocab):
    """Convert email to binary feature vector."""
    words = set(process_email(email))
    return np.array([1 if w in words else 0 for w in vocab], dtype=float)

# Tiny dataset to illustrate
spam_emails = [
    "Buy now! Free medication click here. Limited dollar prize.",
    "Free free free! Buy lowest prices now. Dollar dollar click!",
    "Medication buy now click for prize. Lowest ever free!",
    "Click now free prize dollar buy lowest medication!",
]
ham_emails = [
    "Can we schedule a meeting for the project? Best regards.",
    "Meeting tomorrow to discuss the schedule and project details.",
    "Please schedule time for the project review. Regards John.",
    "Let us schedule a project meeting. Thank you so much.",
]

X_email = np.vstack([email_to_features(e, vocab) for e in spam_emails + ham_emails])
Y_email = np.array([1]*4 + [0]*4)

print("Feature matrix shape:", X_email.shape)  # (8, 13)
print("\nFeature vector for first spam:")
print(dict(zip(vocab, X_email[0].astype(int))))

# Train SVM (as used in Practice L13)
clf = SVC(C=1.0, kernel='linear')
clf.fit(X_email, Y_email)
preds = clf.predict(X_email)
print(f"\nTraining accuracy: {np.mean(preds == Y_email)*100:.0f}%")

---
## Summary

### K-Means:
```python
# E-step: assign to nearest centroid
for i in range(m):
    dists = [(X[i]-c)@(X[i]-c) for c in centroids]
    idx[i] = np.argmin(dists)

# M-step: update centroids as cluster means
for k in range(K):
    centroids[k] = X[idx==k].mean(axis=0)
```

### EM / GMM:
```python
# E-step: compute soft responsibilities
gamma[:, k] = pi[k] * gaussian_pdf(X, mu[k], sigma[k])
gamma /= gamma.sum(axis=1, keepdims=True)  # normalize

# M-step: update parameters using weighted statistics
Nk = gamma.sum(axis=0)
pi = Nk / m
mu = (gamma.T @ X) / Nk[:, np.newaxis]
sigma[k] = weighted_covariance(X, gamma[:, k], mu[k])
```

### The Big Picture:

| Algorithm | Has Labels | Models | Key Idea |
|---|---|---|---|
| Linear Regression | Yes | Continuous output | Minimize MSE |
| Logistic Regression | Yes | Binary probability | Sigmoid + cross-entropy |
| Neural Networks | Yes | Any function | Stacked logistic layers |
| PCA | No | Data structure | Maximum variance projection |
| K-Means | No | Hard clusters | Minimize intra-cluster distance |
| GMM/EM | No | Soft clusters | Maximize data likelihood |

Every algorithm in this course is about fitting a model to data — they differ in:
- What kind of output (continuous, probability, cluster)
- How they measure error (MSE, cross-entropy, distortion, likelihood)
- How they optimize (gradient descent, EM, eigendecomposition)